PRE-TRAINING LLM 

> Load Data

In [68]:
with open("C:/Users/ASUS/Documents/GitHub/LLM-from-Scratch/J. K. Rowling - Harry Potter 1 - Sorcerer's Stone.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:49])

Total number of character: 439742
Harry Potter and the Sorcerer's Stone


CHAPTER O


> ## 1. Tokenization

In [69]:
import re
preprocessed = re.split(r'([,.:;?_!"()\']|-|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:9])

103826
['Harry', 'Potter', 'and', 'the', 'Sorcerer', "'", 's', 'Stone', 'CHAPTER']


In [70]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
vocab = {token:integer for integer,token in enumerate(all_words)}

print(vocab_size)

6667


> 1.1. TokenizerV1 (word-based tokenizer without out of vocabulary handling)

In [71]:
import re

class SimpleTokenizerV1:
    # Main function of the tokenizer
    def __init__(self, vocab):
        self.str_to_int = vocab # Create a mapping from string to integer (encoding)
        self.int_to_str = {i:s for s,i in vocab.items()} # Inverse mapping from integer to string (decoding)
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|-|\s)', text) # Split the text into tokens based on the specified delimiters
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ] # Remove leading and trailing whitespace from each token and filter out empty tokens
        ids = [self.str_to_int[s] for s in preprocessed] # Convert each token to its corresponding integer ID using the mapping
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids]) # Convert each integer ID back to its corresponding token
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

An error occured because of Out of Vocabulary problem (Lowokwaru isn't part of the vocabulary)

In [72]:
tokenizer1 = SimpleTokenizerV1(vocab)

text = "Hermione came from Lowokwaru"
tokenizer1.encode(text)

KeyError: 'Lowokwaru'

> 1.2. TokenizerV2 (word-based tokenizer with out of vocabulary handling)

- Step 1: Replace unknown words by <|unk|> tokens
    
- Step 2: Replace spaces before the specified punctuations

In [73]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [74]:
class SimpleTokenizerV2:
    # Main function of the tokenizer
    def __init__(self, vocab):
        self.str_to_int = vocab # Create a mapping from string to integer (encoding)
        self.int_to_str = { i:s for s,i in vocab.items()} # Inverse mapping from integer to string (decoding)
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) # Split the text into tokens based on the specified delimiters, including double hyphens
        preprocessed = [item.strip() for item in preprocessed if item.strip()] # Remove leading and trailing whitespace from each token and filter out empty tokens
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ] # Replace tokens that are not in the vocabulary with a special token "<|unk|>"

        ids = [self.str_to_int[s] for s in preprocessed] # Convert each token to its corresponding integer ID using the mapping
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids]) # Convert each integer ID back to its corresponding token
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Special context token added to handle out of vocabulary problem

In [75]:
tokenizer2 = SimpleTokenizerV2(vocab)

text = "Hermione came from Lowokwaru"

tokenizer2.decode(tokenizer2.encode(text))

'Hermione came from <|unk|>'

> 1.3. Sub words-based Tokenizer (BPE Tokenization using tiktoken)

In [76]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.11.0


In [77]:
tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

116725


BPE can tokenize "Lowokwaru" without a problem because it's a sub words-based tokenizer and somehow has find a way to construct the word "Lowokwaru"

In [80]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hermione came from Lowokwaru"

tokenizer.decode(tokenizer.encode(text))

'Hermione came from Lowokwaru'

In [82]:
# Let's check which token are used

integers = tokenizer.encode(text)
print(integers)

strings = tokenizer.decode(integers)
print(strings)

[48523, 7935, 1625, 422, 7754, 482, 5767, 84]
Hermione came from Lowokwaru
